# Koddan Kasa · Levha 19 · Replay öğrenme değildir

Videodaki deney: SO-101 kolu (MuJoCo Menagerie modeli) bir küpü **bir kez** alıp kaldırıyor; her 2 ms'de gönderilen 6 komut (5 eklem + çene) kaydediliyor. Sonra aynı komutlar **birebir** tekrar oynatılıyor (replay), ama küp 0,5–5 cm kaydırılmış, 8 yönde.

Bizim sonuçlar (MuJoCo 3.17): replay **27/81** · 0,5 cm 8/8 · 1 cm 7/8 · 2 cm 3/8 · 3 cm 1/8 · 4,5–5 cm 0/8. Küpün yerini okuyup yeniden nişan alan kural (geri besleme, **öğrenme değil**): **81/81**.

Başarı fiziksel: küp yalnız sürtünmeyle tutuluyor (yapıştırma/weld yok). Ölçüt: hareketin sonunda küpün altı ≥ 4 cm yukarıda ve küp iki çeneye de değiyor.

## 1 · Kurulum

In [ ]:
!pip -q install mujoco==3.17.0
!git clone -q --depth 1 https://github.com/google-deepmind/mujoco_menagerie /content/mujoco_menagerie 2>/dev/null || true

In [ ]:
import json, math, os
from pathlib import Path
import mujoco, numpy as np
print("mujoco", mujoco.__version__, "numpy", np.__version__)
# Colab: /content/mujoco_menagerie · yerelde MENAGERIE ortam değişkeni
MENAGERIE = Path(os.environ.get("MENAGERIE", "/content/mujoco_menagerie"))
SO101 = MENAGERIE / "robotstudio_so101/so101.xml"
assert SO101.exists(), SO101

## 2 · Sahne, IK, kayıt ve oynatma

`build` kolu, zemini ve 3 cm'lik küpü kurar. `ik` parmak ucunu (site `gripperframe`) küpün üstüne, çene aşağı bakacak şekilde yerleştiren eklem açılarını bulur. `plan` bu açılardan 4,5 sn'lik komut listesini (2250 × 6) yazar. `run` bir komut listesini adım adım oynatır ve başarıyı **temaslardan** ölçer.

In [ ]:
DT = 0.002  # MuJoCo varsayılan adımı; menagerie dosyası 5 ms diyor, temaslar için 2 ms seçildi
CUBE_HALF = 0.015  # 3 cm küp
CUBE_MASS = 0.03
CUBE_HOME = np.array([0.22, 0.0])  # kaydın yapıldığı yer (taban merkezinden 22 cm ileride)
OFFSETS_CM = [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]
DIRS_DEG = [0, 45, 90, 135, 180, 225, 270, 315]  # 0 = ileri (+x, kolun uzandığı yön), 90 = sola (+y)
ARM = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll"]
G_OPEN, G_CLOSE = 1.2, -0.17  # çene komutu (rad): açık ≈ 6 cm aralık, kapalı = sınır (servo kuvvet sınırıyla sıkar)
PRE_M, LIFT_M, SUCCESS_LIFT_M = 0.055, 0.07, 0.04  # top-down poses reach ≈ 9 cm high at 22 cm (wrist_flex limit)

HOME_Q = np.array([0.0, -1.2, 0.5, 1.2, 1.68])  # toplu bekleme pozu (kol kadrajdan taşmasın), küpün üstünde-gerisinde

# zaman çizelgesi (s): bekle → küpün üstüne → in → kapat → kaldır → bekle
T = dict(settle=0.3, pre=1.5, down=2.3, close=2.9, lift=3.9, hold=4.5)


def build(cube_xy, with_floor_vis=True):
    spec = mujoco.MjSpec.from_file(str(SO101))
    spec.option.timestep = DT
    wb = spec.worldbody
    wb.add_geom(name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[2, 2, 0.05], rgba=[0.3, 0.3, 0.3, 1])
    b = wb.add_body(name="cube", pos=[cube_xy[0], cube_xy[1], CUBE_HALF])
    b.add_freejoint()
    b.add_geom(name="cube_geom", type=mujoco.mjtGeom.mjGEOM_BOX, size=[CUBE_HALF] * 3, mass=CUBE_MASS,
               friction=[1.0, 0.03, 0.003], condim=4, solref=[0.01, 1], rgba=[0.86, 0.12, 0.09, 1])
    m = spec.compile()
    return spec, m, mujoco.MjData(m)


def ids(m):
    jaw_fixed, jaw_moving = set(), set()
    for g in range(m.ngeom):
        body = mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_BODY, m.geom_bodyid[g])
        if m.geom_contype[g] == 0 and m.geom_conaffinity[g] == 0:
            continue
        if body == "gripper":
            jaw_fixed.add(g)
        elif body == "moving_jaw_so101_v1":
            jaw_moving.add(g)
    return dict(cube=m.geom("cube_geom").id, cube_body=m.body("cube").id, site=m.site("gripperframe").id,
                qadr=[m.joint(j).qposadr[0] for j in ARM], dadr=[m.joint(j).dofadr[0] for j in ARM],
                act=[m.actuator(j).id for j in ARM], grip=m.actuator("gripper").id, fixed=jaw_fixed, moving=jaw_moving,
                cube_q=m.joint(m.body("cube").jntadr[0]).qposadr[0])


def ik(m, d, I, target, x_axis=(0, 0, -1), z_axis=(0, 1, 0), q0=None, iters=300):
    """Damped least squares on the 5 arm joints: gripperframe site at `target`, its x axis (approach) pointing down,
    its z axis along world y (jaws close along x, the cube's faces). Kinematics only (qpos, mj_kinematics)."""
    dd = mujoco.MjData(m)
    dd.qpos[:] = d.qpos if q0 is None else q0
    jp, jr = np.zeros((3, m.nv)), np.zeros((3, m.nv))
    xa, za = np.asarray(x_axis, float), np.asarray(z_axis, float)
    for _ in range(iters):
        mujoco.mj_kinematics(m, dd)
        mujoco.mj_comPos(m, dd)
        R = dd.site_xmat[I["site"]].reshape(3, 3)
        ep = np.asarray(target) - dd.site_xpos[I["site"]]
        er = 0.3 * (np.cross(R[:, 0], xa) + 0.5 * np.cross(R[:, 2], za))
        e = np.concatenate([ep, er])
        if np.linalg.norm(ep) < 2e-4 and np.linalg.norm(er) < 2e-3:
            break
        mujoco.mj_jacSite(m, dd, jp, jr, I["site"])
        J = np.vstack([jp, 0.3 * jr])[:, I["dadr"]]
        dq = J.T @ np.linalg.solve(J @ J.T + 1e-4 * np.eye(6), e)
        for k, a in enumerate(I["qadr"]):
            lo, hi = m.jnt_range[m.dof_jntid[I["dadr"][k]]]
            dd.qpos[a] = np.clip(dd.qpos[a] + 0.7 * dq[k], lo, hi)
    mujoco.mj_kinematics(m, dd)
    return np.array([dd.qpos[a] for a in I["qadr"]]), float(np.linalg.norm(np.asarray(target) - dd.site_xpos[I["site"]]))


def minjerk(u):
    u = min(1.0, max(0.0, u))
    return u ** 3 * (10 - 15 * u + 6 * u * u)


def plan(m, d, I, cube_xy):
    """Joint waypoints for a top-down pick of a cube at cube_xy; returns ctrl(t) for every physics step."""
    home = HOME_Q.copy()
    zc = CUBE_HALF
    tip = 0.004  # fingertip site this far below the cube centre (pads cover the cube's middle)
    q_down, e2 = ik(m, d, I, [cube_xy[0], cube_xy[1], zc - tip], iters=1500)
    seed = d.qpos.copy()
    seed[I["qadr"]] = q_down  # same elbow branch for all three poses
    q_pre, e1 = ik(m, d, I, [cube_xy[0], cube_xy[1], zc + PRE_M], q0=seed, iters=1500)
    q_lift, e3 = ik(m, d, I, [cube_xy[0], cube_xy[1], zc + LIFT_M], q0=seed, iters=1500)
    n = int(round(T["hold"] / DT))
    ctrl = np.zeros((n, 6))
    for i in range(n):
        t = (i + 1) * DT
        if t < T["settle"]:
            q, g = home, G_OPEN
        elif t < T["pre"]:
            q, g = home + (q_pre - home) * minjerk((t - T["settle"]) / (T["pre"] - T["settle"])), G_OPEN
        elif t < T["down"]:
            q, g = q_pre + (q_down - q_pre) * minjerk((t - T["pre"]) / (T["down"] - T["pre"])), G_OPEN
        elif t < T["close"]:
            q, g = q_down, G_OPEN + (G_CLOSE - G_OPEN) * minjerk((t - T["down"]) / (T["close"] - T["down"]))
        elif t < T["lift"]:
            q, g = q_down + (q_lift - q_down) * minjerk((t - T["close"]) / (T["lift"] - T["close"])), G_CLOSE
        else:
            q, g = q_lift, G_CLOSE
        ctrl[i, :5] = q
        ctrl[i, 5] = g
    return ctrl, max(e1, e2, e3)


def run(cube_xy, ctrl, keep=False):
    """Plays a ctrl sequence step by step with the cube at cube_xy. Returns outcome + (optionally) the qpos trace."""
    _, m, d = build(cube_xy)
    I = ids(m)
    qs, contacts = [], []
    d.qpos[I["qadr"]] = HOME_Q  # robot starts at rest in the home pose, jaw open
    d.qpos[m.joint("gripper").qposadr[0]] = G_OPEN
    mujoco.mj_forward(m, d)
    start = d.xpos[I["cube_body"]].copy()
    first_touch = None
    for i, c in enumerate(ctrl):
        d.ctrl[I["act"]] = c[:5]
        d.ctrl[I["grip"]] = c[5]
        mujoco.mj_step(m, d)
        if first_touch is None:
            for k in range(d.ncon):
                con = d.contact[k]
                if I["cube"] in (con.geom1, con.geom2):
                    other = con.geom2 if con.geom1 == I["cube"] else con.geom1
                    if other in I["fixed"] or other in I["moving"]:
                        first_touch = round((i + 1) * DT, 3)
                        break
        if keep:
            qs.append(d.qpos.copy())
    fixed = moving = False
    for k in range(d.ncon):
        con = d.contact[k]
        if I["cube"] in (con.geom1, con.geom2):
            other = con.geom2 if con.geom1 == I["cube"] else con.geom1
            fixed |= other in I["fixed"]
            moving |= other in I["moving"]
    end = d.xpos[I["cube_body"]].copy()
    bottom = end[2] - CUBE_HALF
    lifted = bottom >= SUCCESS_LIFT_M
    moved = float(np.linalg.norm(end[:2] - start[:2]))
    ok = bool(lifted and fixed and moving)
    if ok:
        kind = "tuttu"
    elif moved < 0.002 and abs(end[2] - start[2]) < 0.002 and first_touch is None:
        kind = "ıskaladı"
    elif first_touch is not None and not lifted:
        kind = "itti"
    else:
        kind = "düşürdü" if not lifted else "tek çene"
    rec = dict(success=ok, outcome=kind, lift_cm=round(bottom * 100, 2), moved_cm=round(moved * 100, 2),
               jaw_fixed=bool(fixed), jaw_moving=bool(moving), first_touch_s=first_touch)
    return rec, (np.array(qs) if keep else None), m


def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (round(max(0, c - h), 3), round(min(1, c + h), 3))


def sweep(rec_ctrl, mode, keep_pairs=()):
    rows, traces = [], {}
    for r in OFFSETS_CM:
        for a in DIRS_DEG if r > 0 else [0]:
            xy = CUBE_HOME + r / 100 * np.array([math.cos(math.radians(a)), math.sin(math.radians(a))])
            if mode == "replay":
                ctrl, ik_err = rec_ctrl, None
            else:
                _, m0, d0 = build(xy)
                ctrl, ik_err = plan(m0, d0, ids(m0), xy)
            keep = (r, a) in keep_pairs
            res, qs, _ = run(xy, ctrl, keep=keep)
            res.update(offset_cm=r, dir_deg=a, cube_xy=[round(float(v), 4) for v in xy])
            if ik_err is not None:
                res["ik_err_mm"] = round(ik_err * 1000, 2)
            rows.append(res)
            if keep:
                traces[(r, a)] = (qs, ctrl)
            print(f"{mode:7s} {r:3.1f} cm {a:3d}° → {res['outcome']:9s} lift {res['lift_cm']:5.2f} cm moved {res['moved_cm']:5.2f}")
    return rows, traces


def summarize(rows):
    out = []
    for r in OFFSETS_CM:
        sel = [x for x in rows if x["offset_cm"] == r]
        k = sum(x["success"] for x in sel)
        out.append(dict(offset_cm=r, n=len(sel), success=k, rate=round(k / len(sel), 3), wilson95=wilson(k, len(sel)),
                        outcomes={o: sum(x["outcome"] == o for x in sel) for o in sorted({x["outcome"] for x in sel})}))
    return out

## 3 · Kayıt + iki tarama

**Replay:** kayıttaki liste, hiç değiştirilmeden. **Geri beslemeli:** her denemede küpün yeri simülasyondan okunur ve plan yeniden hesaplanır (kamera yok, öğrenme yok).

In [ ]:
_, m, d = build(CUBE_HOME)
I = ids(m)
rec_ctrl, ik_err = plan(m, d, I, CUBE_HOME)
print("kayıt:", rec_ctrl.shape, "adım × komut")
rows_r, _ = sweep(rec_ctrl, "replay")
rows_c, _ = sweep(rec_ctrl, "closed")
sr, sc = summarize(rows_r), summarize(rows_c)
for a, b in zip(sr, sc):
    print(f"{a['offset_cm']:3.1f} cm  replay {a['success']}/{a['n']}  geri besleme {b['success']}/{b['n']}")
print("toplam replay", sum(x['success'] for x in rows_r), "/", len(rows_r), "· geri besleme", sum(x['success'] for x in rows_c), "/", len(rows_c))

## 4 · Yön yön: hangi tarafa kayınca en uzun dayandı? (videodaki sorunun cevabı)

In [ ]:
for a in DIRS_DEG:
    ok = [x['offset_cm'] for x in rows_r if x['dir_deg'] == a and x['success']]
    print(f"{a:3d}°  en uzak başarılı: {max(ok) if ok else 0} cm   sonuçlar:", ''.join('●' if x['success'] else '○' for x in rows_r if x['dir_deg'] == a))
# 0° = ileri (+x), 90° = +y (hareketli çene tarafı), 270° = −y (sabit çene tarafı)

## 5 · Sonucu yaz (videodaki her sayının kaynağı)

In [ ]:
Path("results/19").mkdir(parents=True, exist_ok=True)
res = dict(lesson=19, mujoco=mujoco.__version__, steps=len(rec_ctrl), replay=sr, closed=sc,
           totals=dict(replay=sum(x['success'] for x in rows_r), closed=sum(x['success'] for x in rows_c), n=len(rows_r)))
Path("results/19/result.json").write_text(json.dumps(res, ensure_ascii=False, indent=1))
print(res["totals"])

## 6 · Tarama: kendin değiştir

Tek bir sayıyı değiştir ve replay'in toleransını gör. Örnek: çeneyi daha az aç (`G_OPEN`), ya da küpü büyüt (`CUBE_HALF`). Kayıt her seferinde yeniden alınır.

In [ ]:
def replay_total(g_open=1.2, cube_half=0.015):
    global G_OPEN, CUBE_HALF
    old = (G_OPEN, CUBE_HALF)
    G_OPEN, CUBE_HALF = g_open, cube_half
    try:
        _, m, d = build(CUBE_HOME)
        rec, _ = plan(m, d, ids(m), CUBE_HOME)
        import io, contextlib
        with contextlib.redirect_stdout(io.StringIO()):
            rows, _ = sweep(rec, "replay")
        return sum(x['success'] for x in rows), len(rows)
    finally:
        G_OPEN, CUBE_HALF = old

for g in (0.8, 1.0, 1.2, 1.5):   # ← bu listeyi değiştir
    k, n = replay_total(g_open=g)
    print(f"çene açıklığı {g:.1f} rad → replay {k}/{n}")